# Utility Placement Optimisation

**Learning outcome:** Apply utility placement optimisation through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `base`  
**Expected runtime:** under 2 minutes  
**Optional extras:** `notebook`

## Study question and data

**Study question:** Where should four isothermal hot and cold utility levels be placed at Process and Site hierarchy levels to minimize thermodynamic cost?

## Step 1: Prepare the placement study

This cell opens `chocolate_factory.json` in a `PinchWorkspace` and activates the `baseline` case. `baseline_input` keeps its canonical input for a later check. `retarget_comparison()` tabulates each optimised utility level's allocated duty against the retargeted duty. `search_options` sets small iteration, evaluation, candidate and run limits.

In [ ]:
import pandas as pd

from OpenPinch import PinchWorkspace

workspace = PinchWorkspace(source="chocolate_factory.json", project_name="Site")
problem = workspace.use_case("baseline")
baseline_input = problem.to_problem_json()


def retarget_comparison(evidence, target):
    period = evidence.best.period_results[0]
    rows = []
    for side, levels, utilities in (
        ("hot", period.hot_levels, target.hot_utilities),
        ("cold", period.cold_levels, target.cold_utilities),
    ):
        retargeted = {
            utility.name: float(utility.heat_flow.value) for utility in utilities
        }
        for level in levels:
            name = level.template_key.name
            rows.append(
                {
                    "side": side,
                    "utility": name,
                    "kind": level.kind.value,
                    "supply_degC": level.supply_temperature.value,
                    "target_degC": level.target_temperature.value,
                    "optimizer_duty_kW": level.allocated_duty.value,
                    "retargeted_duty_kW": retargeted.get(name, 0.0),
                    "difference_kW": retargeted.get(name, 0.0)
                    - level.allocated_duty.value,
                    "fallback": level.is_fallback,
                }
            )
    return pd.DataFrame(rows)


search_options = {
    "iteration_limit": 5,
    "evaluation_limit": 10,
    "candidate_limit": 1,
    "run_count": 1,
}

## Step 2: Optimize a Process Zone and build its standard GCC

`utility_placement()` places four isothermal levels per side and no sensible levels for the `Almond` process zone in period `0`. It returns the solved case `process_case`. `process_objective` and `process_fallback_penalty` summarise the best candidate. `workspace.add(..., activate=False)` registers the case without activating it. Check `process_retarget_comparison` and `process_gcc`.

In [ ]:
process_case = problem.target.utility_placement(
    isothermal=4,
    sensible=0,
    zone="Almond",
    period_ids=("0",),
    options=search_options,
)
process_evidence = process_case.utility_placement_result
process_search = process_evidence.termination.model_dump(mode="json")
process_objective = process_evidence.best.aggregate_objective
process_fallback_penalty = process_evidence.best.fallback_penalty
process_utilities = process_case.to_problem_json()["utilities"]
registered_process = workspace.add(
    process_case,
    name="optimized_process_utilities",
    activate=False,
)
process_target = process_case.results.targets[-1]
process_retarget_comparison = retarget_comparison(process_evidence, process_target)
process_summary = process_case.summary_frame()
process_gcc = process_case.plot.grand_composite_curve(zone_name="Almond")

## Step 3: Optimize the Site and build its standard Total Site Profile

This cell repeats the placement without a `zone`, so it optimises the site profile. It uses four isothermal levels and period `0`. `site_objective` and `site_utilities` report the best candidate. The case is registered with `activate=False`, and `baseline_unchanged` confirms the baseline input is untouched. Check `site_retarget_comparison` and `site_tsp`.

In [ ]:
site_case = problem.target.utility_placement(
    isothermal=4,
    sensible=0,
    period_ids=("0",),
    options=search_options,
)
site_evidence = site_case.utility_placement_result
site_search = site_evidence.termination.model_dump(mode="json")
site_objective = site_evidence.best.aggregate_objective
site_utilities = site_case.to_problem_json()["utilities"]
registered_site = workspace.add(
    site_case,
    name="optimized_site_utilities",
    activate=False,
)
baseline_unchanged = workspace.use_case("baseline").to_problem_json() == baseline_input
site_target = site_case.results.targets[-1]
site_retarget_comparison = retarget_comparison(site_evidence, site_target)
site_summary = site_case.summary_frame()
site_tsp = site_case.plot.total_site_profiles()

## Review the result

Review both optimized cases exactly like normal cases: compare the Process utilities on the standard GCC with the Site utilities on the standard Total Site Profile. The Process Utility GCC must not cross the Process GCC. The comparison tables place optimizer-evidence and cached allocation duties side by side; a zero difference shows exact replay.

In [ ]:
from IPython.display import display

display(process_objective)
display(process_fallback_penalty)
display(process_search)
display(process_retarget_comparison)
display(process_summary)
display(process_gcc)
display(site_objective)
display(site_search)
display(site_retarget_comparison)
display({"baseline_unchanged": baseline_unchanged})
display(site_summary)
display(site_tsp)

## Interpret the result

Compare the Process result against its direct GCC and the Site result against its Total Site Profile. Candidate duties come from those exact ordinary target workflows; they are not independent optimizer decisions, and a requested level may be unused. Inspect physical entropy generation from the balanced composite curves using the signed Q / T limit for the isothermal intervals; CP * ln(T_out / T_in) in kelvin applies when adapting the workflow to sensible utilities. Confirm that final allocation reproduces every selected duty and that the Utility GCC does not cross the Process GCC.

## Adapt this template

Replace the sample with validated plant data, apply defensible temperature bounds, and increase the optimizer limits before making an engineering decision.